# Correr el agente en Kaggle (2×T4)

Clona la rama, instala con uv las versiones fijas y corre `COMANDO` con `correr_kaggle.py`. Al final queda un zip de `corridas/` en el panel *Output*.

Antes: *Settings* → Accelerator **GPU T4 x2**, Internet **on**, y en *Add-ons* → *Secrets* un secreto `HF_TOKEN` (lectura de Hugging Face). Ver `kaggle/README.md`.

In [ ]:
# Parámetros: lo único que se cambia entre corridas.
RAMA = "juan/agente-prompt"
COMMIT = ""  # vacío: la punta de la rama; o un hash para fijarlo
COMANDO = "optimizacion.optimizar --nombre ensayo-kaggle-01 --limite-casos 3 --max-evaluaciones 1 --normas original"
LOTE = ""  # vacío: el de configuracion.json; bajarlo si da CUDA out of memory

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["PYTHONUTF8"] = "1"
REPO = "/tmp/repo"  # fuera de /kaggle/working, para que el .venv no cuente como salida

def sh(comando):
    print("$", comando, flush=True)
    subprocess.run(comando, shell=True, check=True, cwd=REPO if os.path.isdir(REPO) else None)

sh("nvidia-smi")
if not os.path.isdir(REPO):
    sh(f"git clone -q -b {RAMA} https://github.com/genai-grupo-2/emergency-admission-agent.git {REPO}")
if COMMIT:
    sh(f"git checkout -q {COMMIT}")
sh("git log --oneline -1 && git status --short")

In [ ]:
# Mismas versiones que la imagen de modal_app.py. Si la rueda de torch por defecto no ve la GPU (driver viejo),
# se prueba con las de CUDA 12.x. La que quedó se anota en corridas/entornos/.
sh("pip install -q uv")
sh("uv sync --locked -q")
sh("uv pip install -q torch==2.14.1 transformers==5.20.2 accelerate==1.15.0")
ver = ".venv/bin/python -c \"import torch; assert torch.cuda.device_count() >= 2, torch.cuda.device_count(); print(torch.__version__, torch.version.cuda)\""
for indice in [None, "cu128", "cu126"]:
    if indice:
        sh(f"uv pip install -q --reinstall torch==2.14.1 --index-url https://download.pytorch.org/whl/{indice}")
    if subprocess.run(ver, shell=True, cwd=REPO).returncode == 0:
        break
else:
    raise RuntimeError("torch no ve las dos GPU: revisar el accelerator (GPU T4 x2) y la salida de nvidia-smi")

In [ ]:
lote = f" --lote {LOTE}" if LOTE else ""
sh(f".venv/bin/python correr_kaggle.py --comando \"{COMANDO}\"{lote}")